# ML 데이터 준비Purpose: validate the attached shared Dataset and build deterministic row-based ML views.> Warning: this is an oracle/sanity-only synthetic-data benchmark, not real-device or medical-performance evidence.

In [ ]:
from __future__ import annotationsimport hashlibimport jsonimport refrom pathlib import Pathfrom typing import Any, Iterableimport pandas as pdSERIES_ID = "mvp3-oracle-v1"EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}DATA_STATUS = "oracle/sanity"REAL_ACCURACY_STATUS = "NOT VERIFIED"DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"RUN_TRAINING = FalseRUN_LOCKED_TEST = FalseML_OUTPUT_ROOT = Path("/kaggle/working/goal15_ml_view")ORACLE_FEATURE_DENYLIST = (    "active_target_",    "hard_negative_id",    "hard_negative_type",    "artifact_schedule_id",    "participant_truth_baseline",    "event_intensity_truth",)STABLE_KEYS = ["person_key", "canonical_time"]

## 1. Validate the immutable shared DatasetThe notebook reads only flat Dataset files. It never reads hidden truth directories or writes to the attached input.

In [ ]:
def sha256_file(path: Path) -> str:    digest = hashlib.sha256()    with path.open("rb") as handle:        for chunk in iter(lambda: handle.read(1024 * 1024), b""):            digest.update(chunk)    return digest.hexdigest()def _validate_sha256(value: Any, field: str) -> str:    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:        raise ValueError(f"invalid SHA-256 for {field}")    return valuedef _validate_file_hash(dataset_root: Path, filename: str, expected: Any) -> None:    path = dataset_root / filename    if not path.is_file():        raise FileNotFoundError(f"missing dataset file: {filename}")    expected_hash = _validate_sha256(expected, filename)    if sha256_file(path) != expected_hash:        raise ValueError(f"hash mismatch for {filename}")def resolve_kaggle_dataset_root(input_root: Path = Path("/kaggle/input")) -> Path:    required_prefixes = ("prepared__", "outcomes__", "registry__")    candidates = [input_root, *sorted(path for path in input_root.iterdir() if path.is_dir())]    for candidate in candidates:        names = {path.name for path in candidate.iterdir()}        if all(any(name.startswith(prefix) for name in names) for prefix in required_prefixes):            return candidate    raise FileNotFoundError("attached Dataset is missing prepared__, outcomes__, or registry__ files")def load_split_registry(dataset_root: Path) -> pd.DataFrame:    split_path = dataset_root / "registry__splits.parquet"    if not split_path.is_file():        raise FileNotFoundError(f"missing split registry: {split_path}")    split = pd.read_parquet(split_path)    required_columns = {"person_key", "split_role"}    missing = required_columns.difference(split.columns)    if missing:        raise ValueError(f"split registry missing columns: {sorted(missing)}")    validate_split_contract(split)    return split[["person_key", "split_role"]].drop_duplicates()def validate_split_contract(split: pd.DataFrame) -> None:    counts = split.groupby("split_role")["person_key"].nunique().to_dict()    if counts != EXPECTED_SPLIT_COUNTS:        raise ValueError(f"split mismatch: {counts}")    overlaps = [        set(split.loc[split["split_role"] == role, "person_key"])        for role in EXPECTED_SPLIT_COUNTS    ]    if any(overlaps[i] & overlaps[j] for i in range(3) for j in range(i + 1, 3)):        raise ValueError("person leakage across split roles")def validate_manifest_hashes(dataset_root: Path) -> dict[str, str]:    manifest_names = ("prepared__manifest.json", "outcomes__manifest.json", "registry__manifest.json")    manifests: dict[str, dict[str, Any]] = {}    hashes: dict[str, str] = {}    for name in manifest_names:        manifest_path = dataset_root / name        if not manifest_path.is_file():            raise FileNotFoundError(f"missing manifest: {name}")        manifest = json.loads(manifest_path.read_text())        declared_series = manifest.get("series_id") or manifest.get("dataset_id")        if declared_series is not None and declared_series != SERIES_ID:            raise ValueError(f"unexpected series in {name}: {declared_series}")        manifests[name] = manifest        hashes[name] = sha256_file(manifest_path)    prepared = manifests["prepared__manifest.json"]    people = prepared.get("people")    if not isinstance(people, list) or not people:        raise ValueError("prepared manifest must declare people")    for person in people:        if not isinstance(person, dict) or not isinstance(person.get("dataset_id"), str):            raise ValueError("prepared manifest has invalid person entry")        _validate_sha256(person.get("logical_hash"), f"logical hash for {person['dataset_id']}")        person_path = dataset_root / f"prepared__people__{person['dataset_id']}.parquet"        if not person_path.is_file():            raise FileNotFoundError(f"missing dataset file: {person_path.name}")    for hash_field, filename in (        ("personal_baseline_sha256", "prepared__personal_baseline.parquet"),        ("source_split_sha256", "registry__splits.parquet"),    ):        if hash_field not in prepared:            raise ValueError(f"missing required hash: {hash_field}")        _validate_file_hash(dataset_root, filename, prepared[hash_field])    outcomes = manifests["outcomes__manifest.json"]    outcome_files = outcomes.get("files")    if not isinstance(outcome_files, dict) or not outcome_files:        raise ValueError("outcomes manifest must declare file hashes")    for relative_name, expected_hash in outcome_files.items():        if not isinstance(relative_name, str):            raise ValueError("outcomes manifest has invalid filename")        _validate_file_hash(dataset_root, f"outcomes__{relative_name}", expected_hash)    registry = manifests["registry__manifest.json"]    if "records_sha256" not in registry:        raise ValueError("missing required hash: records_sha256")    _validate_file_hash(dataset_root, "registry__registry.jsonl", registry["records_sha256"])    registry_path = dataset_root / "registry__registry.jsonl"    if not registry_path.is_file():        raise FileNotFoundError("missing dataset file: registry__registry.jsonl")    registry_hashes = {}    for line in registry_path.read_text().splitlines():        record = json.loads(line)        dataset_id = record.get("dataset_id")        logical_hash = record.get("logical_hash")        if not isinstance(dataset_id, str):            raise ValueError("registry record missing dataset_id")        registry_hashes[dataset_id] = _validate_sha256(logical_hash, f"logical hash for {dataset_id}")    for person in people:        if registry_hashes.get(person["dataset_id"]) != person["logical_hash"]:            raise ValueError(f"logical hash mismatch for {person['dataset_id']}")    return hashesdef assert_no_truth_leakage(columns: Iterable[str]) -> None:    leaked = [        column        for column in columns        if any(column == token or column.startswith(token) for token in ORACLE_FEATURE_DENYLIST)    ]    if leaked:        raise ValueError(f"truth leakage columns: {sorted(leaked)}")def _drop_oracle_columns(frame: pd.DataFrame) -> pd.DataFrame:    denied = [        column for column in frame.columns        if any(column == token or column.startswith(token) for token in ORACLE_FEATURE_DENYLIST)    ]    return frame.drop(columns=denied, errors="ignore")

## 2. Build deterministic ML row viewsTraining keeps every positive and hard negative, then takes at most three deterministically ordered baseline rows per positive. Validation and locked test keep their full 1 Hz timelines.

In [ ]:
def _load_prepared_rows(dataset_root: Path) -> pd.DataFrame:    paths = sorted(dataset_root.glob("prepared__people__*.parquet"))    if not paths:        raise FileNotFoundError("missing prepared__people__*.parquet")    prepared = pd.concat((pd.read_parquet(path) for path in paths), ignore_index=True)    missing = set(STABLE_KEYS).difference(prepared.columns)    if missing:        raise ValueError(f"prepared rows missing stable keys: {sorted(missing)}")    assert_no_truth_leakage(prepared.columns)    return prepareddef _load_outcome_labels(dataset_root: Path) -> pd.DataFrame:    event_path = dataset_root / "outcomes__outcome_events.parquet"    if not event_path.is_file():        raise FileNotFoundError(f"missing outcome labels: {event_path}")    labels = pd.read_parquet(event_path)    missing = set(STABLE_KEYS).difference(labels.columns)    if missing:        raise ValueError(f"outcome labels missing stable keys: {sorted(missing)}")    if labels.duplicated(STABLE_KEYS).any():        raise ValueError("outcome event labels must be unique per person/time")    return labelsdef _deterministic_order(frame: pd.DataFrame) -> pd.DataFrame:    ordered = frame.copy()    ordered["_sample_hash"] = pd.util.hash_pandas_object(        ordered[STABLE_KEYS], index=False, categorize=True    )    return ordered.sort_values(["_sample_hash", *STABLE_KEYS], kind="mergesort")def build_ml_role_view(    prepared: pd.DataFrame,    labels: pd.DataFrame,    split: pd.DataFrame,    split_role: str,) -> pd.DataFrame:    if split_role not in EXPECTED_SPLIT_COUNTS:        raise ValueError(f"unknown split role: {split_role}")    assert_no_truth_leakage(prepared.columns)    role_people = split.loc[split["split_role"] == split_role, ["person_key"]]    feature_rows = prepared.merge(role_people, on="person_key", how="inner", validate="many_to_one")    view = feature_rows.merge(labels, on=STABLE_KEYS, how="left", validate="one_to_one")    label_columns = [column for column in labels.columns if column not in STABLE_KEYS]    view[label_columns] = view[label_columns].fillna(0)    event_columns = [column for column in ("event_label", "event_binary", "label") if column in view]    hard_negative_columns = [        column for column in ("hard_negative", "is_hard_negative", "hard_negative_id", "hard_negative_type")        if column in view    ]    if not event_columns:        raise ValueError("outcome events need event_label, event_binary, or label")    positive_mask = view[event_columns].astype(bool).any(axis=1)    hard_negative_mask = pd.Series(False, index=view.index)    for column in hard_negative_columns:        values = view[column]        if column in {"hard_negative_id", "hard_negative_type"}:            hard_negative_mask |= values.notna() & values.ne(0) & values.astype(str).str.strip().ne("")        else:            hard_negative_mask |= values.astype(bool)    if split_role != "train":        sanitized = _drop_oracle_columns(view)        assert_no_truth_leakage(sanitized.columns)        return sanitized.sort_values(STABLE_KEYS, kind="mergesort").reset_index(drop=True)    required_rows = view.loc[positive_mask | hard_negative_mask]    baseline_candidates = view.loc[~(positive_mask | hard_negative_mask)]    baseline_limit = 3 * int(positive_mask.sum())    sampled_baselines = _deterministic_order(baseline_candidates).head(baseline_limit)    sampled = pd.concat([required_rows, sampled_baselines], ignore_index=True)    internal_columns = [column for column in sampled if column.startswith("_sample_")]    sampled = sampled.drop(columns=internal_columns, errors="ignore")    sampled = _drop_oracle_columns(sampled)    assert_no_truth_leakage(sampled.columns)    return sampled.sort_values(STABLE_KEYS, kind="mergesort").reset_index(drop=True)def write_ml_view_manifest(    output_root: Path,    view_paths: dict[str, Path],    source_dataset_hash: str,    split_hash: str,) -> Path:    files: dict[str, dict[str, Any]] = {}    for split_role, view_path in view_paths.items():        view = pd.read_parquet(view_path)        files[split_role] = {            "path": view_path.name,            "sha256": sha256_file(view_path),            "row_count": len(view),            "columns": list(view.columns),        }    manifest_path = output_root / "view_manifest.json"    manifest_path.write_text(json.dumps({        "series_id": SERIES_ID,        "data_status": DATA_STATUS,        "source_dataset_hash": source_dataset_hash,        "split_hash": split_hash,        "files": files,    }, indent=2, sort_keys=True) + "\n")    return manifest_pathdef build_all_ml_views() -> Path:    dataset_root = resolve_kaggle_dataset_root()    manifest_hashes = validate_manifest_hashes(dataset_root)    split = load_split_registry(dataset_root)    prepared = _load_prepared_rows(dataset_root)    labels = _load_outcome_labels(dataset_root)    ML_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)    view_paths: dict[str, Path] = {}    for split_role in EXPECTED_SPLIT_COUNTS:        view = build_ml_role_view(prepared, labels, split, split_role)        view_path = ML_OUTPUT_ROOT / f"{split_role}.parquet"        view.to_parquet(view_path, index=False, compression="zstd")        view_paths[split_role] = view_path    source_dataset_hash = hashlib.sha256(json.dumps(manifest_hashes, sort_keys=True).encode()).hexdigest()    split_hash = sha256_file(dataset_root / "registry__splits.parquet")    return write_ml_view_manifest(ML_OUTPUT_ROOT, view_paths, source_dataset_hash, split_hash)

## 3. Explicit execution gateData preparation remains disabled in the committed notebook.

In [ ]:
RUN_DATA_PREPARATION = Falseif RUN_DATA_PREPARATION:    build_all_ml_views()else:    print("준비 완료: RUN_DATA_PREPARATION=True로 바꿀 때만 데이터를 생성합니다.")